# 6. Natural atomic charges and HOMO energies (Figure 3)

Extracts, from the Gaussian 16 output of each training solvent, the most positive and
most negative natural atomic charge (natural population analysis, NBO 3.1) and the
HOMO energy, and relates them to the experimental yield.

| Output | Corresponding item in the paper |
|---|---|
| `output/nbo.csv` | the extracted descriptors |
| `output/figure_3_nbo_charge.png` | Figure 3 |
| `output/figure_s_pairplot.png` | overview of all pairwise relationships |

**Input:** `qm/qm_nbo_t6311++g/*.out`, `data/solvent.csv`.

Each Gaussian output file is named after the SMILES string of the solvent, which is
how the calculations are matched to the rows of `solvent.csv`.

In [ ]:
import cclib
import pandas as pd
import matplotlib.pyplot as plt
from seaborn import pairplot

from pathlib import Path

# Resolve paths relative to the repository root, so that the notebook runs
# both from notebooks/ and from the repository root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
QM = ROOT / "qm" / "qm_nbo_t6311++g"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

SAVE_DPI = 600
plt.rcParams["figure.dpi"] = 100

out_files = sorted(QM.glob("*.out"))    # sorted, so the result is reproducible
if not out_files:
    raise FileNotFoundError(
        f"No Gaussian output files found in {QM}. "
        "Place the .com/.out pairs of the ten training solvents there."
    )
print(f"{len(out_files)} Gaussian output files found")

## Parse the Gaussian output files

`Path.stem` strips only the final extension, so a filename such as `C(CCl)Cl.out`
yields the SMILES `C(CCl)Cl` intact. (Splitting on the first `.` would truncate any
SMILES containing a period, e.g. a salt or a multi-component system.)

`moenergies` is reported in eV by cclib, and `homos` holds the index of the highest
occupied orbital for each spin; for these closed-shell molecules only index 0 exists.

In [ ]:
records = []
for path in out_files:
    data = cclib.io.ccread(str(path))
    charges = data.getattributes()["atomcharges"]["natural"]

    records.append({
        "smiles": path.stem,
        "max_nbo": max(charges),                              # most positive charge
        "min_nbo": min(charges),                              # most negative charge
        "homo": data.moenergies[0][data.homos[0]],            # HOMO energy in eV
    })

nbo = pd.DataFrame(records)
nbo.to_csv(OUT / "nbo.csv", index=False)
nbo

## Merge with the experimental yields

The merge is an inner join on the SMILES string, so a mismatch between a filename and
`solvent.csv` silently drops the row. The assertion makes such a mismatch visible.

In [ ]:
solvent = pd.read_csv(DATA / "solvent.csv")
data = solvent.merge(nbo, on="smiles", how="inner")

missing = set(nbo["smiles"]) - set(data["smiles"])
assert not missing, f"SMILES present in the QM files but not in solvent.csv: {missing}"

print(f"{len(data)} solvents matched")
data

## Overview of the descriptors

In [ ]:
grid = pairplot(data[["exp_yield", "max_nbo", "min_nbo", "homo"]])
grid.savefig(OUT / "figure_s_pairplot.png", dpi=SAVE_DPI, bbox_inches="tight")
plt.show()

## Figure 3: most negative natural atomic charge versus experimental yield

The most negative natural atomic charge is a proxy for the Lewis basicity of the
solvent. Solvents whose heteroatom carries a large negative charge (EtOH, DMF, DMSO)
give low yields, consistent with coordination to the Lewis acidic scandium centre
competing with the substrate.

In [ ]:
X_LABEL = "min_nbo"

# labelled version, for inspection
fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.scatter(data[X_LABEL], data["exp_yield"], color="#232A34")
for x, y, name in zip(data[X_LABEL], data["exp_yield"], data["name"]):
    ax.annotate(name, (x, y), fontsize=9, xytext=(3, 3), textcoords="offset points")
ax.set(xlabel="Natural atomic charge (most negative)",
       ylabel="Experimental yield [%]")
fig.savefig(OUT / "figure_3_nbo_charge_labelled.png", dpi=SAVE_DPI, bbox_inches="tight")
plt.show()

# unlabelled version, as printed in the paper
fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(data[X_LABEL], data["exp_yield"], color="#232A34")
ax.set(xlabel="Natural atomic charge (min)", ylabel="Experimental yield [%]")
fig.savefig(OUT / "figure_3_nbo_charge.png", dpi=SAVE_DPI, bbox_inches="tight")
plt.show()

The relationship is qualitative rather than quantitative: THF (−0.607, 83%) and DMF
(−0.621, 0%) carry almost the same charge but behave very differently, so the charge
alone does not capture the coordination behaviour. The analysis is presented as a
chemically intuitive rationale for the trend identified by the fingerprint model, not
as a competing predictive descriptor.